# CPS for time series: decisions and benchmarks

Discrete panel-demand example. In addition to horizon-wise calibration, the notebook uses the complete CPS distribution in the Newsvendor solver and compares the decision against economic-loss and tail-risk benchmarks.

In [1]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor

from tinyconformal.series import DiscreteConformalPredictiveSystemTimeSeriesRegressor
from tinyconformal.utils import NewsvendorSolver

## Demand panel

The synthetic panel avoids external dependencies and includes seasonality, trend, and overdispersion. The final 12 months form the test set.

In [2]:
rng = np.random.default_rng(42)
n_series, periods, horizon = 4, 96, 12
dates = pd.date_range("2018-01-01", periods=periods, freq="MS")
rows = []
for item in range(n_series):
    t = np.arange(periods)
    mean = 28 + 5 * item + 0.12 * t + 8 * np.sin(2 * np.pi * t / 12 + item / 3)
    demand = rng.poisson(np.maximum(mean, 1))
    rows.append(pd.DataFrame({"unique_id": f"sku_{item}", "ds": dates, "y": demand}))
df = pd.concat(rows, ignore_index=True)
train = df.groupby("unique_id", group_keys=False).head(periods - horizon).reset_index(drop=True)
test = df.groupby("unique_id", group_keys=False).tail(horizon).reset_index(drop=True)
test.head()

,unique_id,ds,y
0,sku_0,2025-01-01,38
1,sku_0,2025-02-01,34
2,sku_0,2025-03-01,36
3,sku_0,2025-04-01,50
4,sku_0,2025-05-01,39


## Discrete CPS and predictive distribution

Calibration stores signed residuals separately for each forecast-horizon step. The seasonal benchmark repeats demand observed 12 months earlier.

In [3]:
learner = MLForecast(
    models={"RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=3, random_state=42, n_jobs=-1)},
    freq="MS",
    lags=[1, 2, 3, 6, 12],
)
cps = DiscreteConformalPredictiveSystemTimeSeriesRegressor(
    learner=learner, horizon=horizon, n_windows=4, alpha=0.10, minimum=0
).fit(train, static_features=[], n_jobs=1)
forecast, distributions = cps.predict_distribution(h=horizon)
distribution = distributions["RandomForest"]
forecast[["q10_cps", "q50_cps", "q90_cps"]] = distribution.ppf(
    np.broadcast_to([0.10, 0.50, 0.90], (len(forecast), 3))
)
forecast.head()

,unique_id,ds,RandomForest,q10_cps,q50_cps,q90_cps
0,sku_0,2025-01-01,32.577301,24,42,53
1,sku_0,2025-02-01,41.314377,36,46,56
2,sku_0,2025-03-01,45.493124,45,50,65
3,sku_0,2025-04-01,48.635378,44,54,65
4,sku_0,2025-05-01,51.146126,44,54,69


In [4]:
cps.evaluate(test, h=horizon)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,RandomForest,90%,0.1,0.938,27.75,34.833


## Newsvendor solver and marginal benefit

With an underage cost of 8 and an overage cost of 2, the critical fractile is 80%. `optimize_distribution` queries the conformal PPF directly, without interpolating interval endpoints. For discrete demand, `marginal_benefit_distribution` uses the conformal CDF to measure the expected net value of adding each candidate inventory unit.

In [5]:
decision = forecast.copy()
decision["underage_cost"] = 8.0
decision["overage_cost"] = 2.0
decision = NewsvendorSolver.optimize_distribution(
    decision, distribution, underage_cost="underage_cost", overage_cost="overage_cost"
)
decision["Optimal CPS"] = decision.pop("y_optimal")
decision["CPS median"] = decision["q50_cps"]
decision["Random Forest"] = np.maximum(np.rint(decision["RandomForest"]), 0)
decision["Seasonal naive"] = train.groupby("unique_id")["y"].tail(horizon).to_numpy()
decision["y"] = test["y"].to_numpy()
display(decision[["unique_id", "ds", "critical_ratio", "Optimal CPS", "CPS median", "Random Forest", "Seasonal naive", "y"]].head())
marginal_benefit = NewsvendorSolver.marginal_benefit_distribution(
    decision,
    distribution,
    underage_cost="underage_cost",
    overage_cost="overage_cost",
    units=[30, 40, 50, 60],
)
marginal_benefit[["unique_id", "ds", "MB(k=30)", "MB(k=40)", "MB(k=50)", "MB(k=60)"]].head()

,unique_id,ds,critical_ratio,Optimal CPS,CPS median,Random Forest,Seasonal naive,y
0,sku_0,2025-01-01,0.8,49.0,42,33.0,30,38
1,sku_0,2025-02-01,0.8,53.0,46,41.0,50,34
2,sku_0,2025-03-01,0.8,60.0,50,45.0,53,36
3,sku_0,2025-04-01,0.8,58.0,54,49.0,45,50
4,sku_0,2025-05-01,0.8,59.0,54,51.0,50,39


## Economic loss and tail risk

The per-period loss is $c_u\max(y-q,0)+c_o\max(q-y,0)$. Tail risk reports empirical 95% VaR and CVaR, as well as the worst-case loss.

In [6]:
models = ["Optimal CPS", "CPS median", "Random Forest", "Seasonal naive"]

def period_loss(frame, model):
    actual = frame["y"].to_numpy(float)
    order = frame[model].to_numpy(float)
    cu = frame["underage_cost"].to_numpy(float)
    co = frame["overage_cost"].to_numpy(float)
    return cu * np.maximum(actual - order, 0) + co * np.maximum(order - actual, 0)

def economic_loss(frame, model_names):
    return pd.DataFrame({"model": model_names, "economic_loss": [period_loss(frame, m).sum() for m in model_names]}).sort_values("economic_loss")

def tail_risk(frame, model_names, risk_level=0.95):
    rows = []
    for model in model_names:
        loss = period_loss(frame, model)
        var = np.quantile(loss, risk_level)
        tail = loss[loss >= var]
        rows.append({"model": model, "expected_loss": loss.mean(), "VaR_95": var, "CVaR_95": tail.mean(), "worst_loss": loss.max()})
    return pd.DataFrame(rows).sort_values("CVaR_95")

In [7]:
economic_loss(decision, models)

,model,economic_loss
0,Optimal CPS,1142.0
1,CPS median,1714.0
2,Random Forest,2128.0
3,Seasonal naive,2420.0


In [8]:
tail_risk(decision, models)

,model,expected_loss,VaR_95,CVaR_95,worst_loss
0,Optimal CPS,23.791667,48.0,62.400000,88.0
1,CPS median,35.708333,96.0,114.666667,160.0
3,Seasonal naive,50.416667,117.2,136.000000,152.0
2,Random Forest,44.333333,125.2,173.333333,216.0
